# Deep Learning Statistical Arbitrage — Experiment Analysis Dashboard

This notebook provides interactive exploration, metric comparison, risk-adjusted performance evaluation, and visualization across all experimental runs stored in `results/`.

**Key Metrics Covered:**
- Annualized Sharpe Ratio, Sortino Ratio, and Calmar Ratio
- Maximum Drawdown & Drawdown Trajectories
- Daily L1 Turnover & Net-of-Cost Sharpe (5 bps, 10 bps, 20 bps)
- Breakeven Transaction Costs
- Strategy Cross-Correlations & Return Distributions

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure project root is in path
project_root = Path.cwd().resolve()
if project_root.name == "analysis":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis import (
    scan_runs,
    generate_comparison_table,
    format_markdown_table,
    plot_cumulative_returns,
    plot_rolling_sharpe,
    plot_drawdowns,
    plot_return_distributions,
    plot_correlation_matrix,
    plot_turnover_vs_sharpe,
    plot_executive_dashboard,
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
print(f"Project root: {project_root}")

## 1. Load Experiment Runs and Aggregate Metrics

Scans `results/` for directories containing `config.yaml`, `metrics.json`, and `returns.csv`.

In [ ]:
results_dir = project_root / "results"
runs = scan_runs(results_dir)
print(f"Loaded {len(runs)} runs from {results_dir}")

summary_df = generate_comparison_table(runs)
summary_df

## 2. Cumulative Out-of-Sample Performance and Drawdowns

Shows total equity trajectory alongside underwater drawdown depths.

In [ ]:
returns_dict = {r["run_name"]: r["returns"] for r in runs if r["returns"] is not None}
if returns_dict:
    fig = plot_cumulative_returns(returns_dict, show_drawdown=True)
    plt.show()
else:
    print("No return series found yet. Run experiments first.")

## 3. Rolling Sharpe Ratio Stability

Evaluates rolling 63-day (~quarterly) annualized Sharpe ratio to detect regime-dependent performance degradation.

In [ ]:
if returns_dict:
    fig = plot_rolling_sharpe(returns_dict, window=63)
    plt.show()

## 4. Friction Analysis: Turnover vs. Sharpe & Breakeven Costs

Evaluates whether strategies survive realistic transaction costs (5 bps to 20 bps).

In [ ]:
if not summary_df.empty and "daily_turnover" in summary_df.columns:
    fig = plot_turnover_vs_sharpe(summary_df)
    plt.show()
    
    # Display net performance breakdown
    cols = [c for c in ["run_name", "annualized_sharpe", "daily_turnover", "net_sharpe_5bps", "net_sharpe_10bps", "breakeven_cost_bps"] if c in summary_df.columns]
    display(summary_df[cols])

## 5. Strategy Cross-Correlations

Checks if different deep learning models or baselines (e.g. CNN-Transformer vs Raw FFN vs Reversal) generate orthogonal return streams suitable for ensembling.

In [ ]:
if len(returns_dict) >= 2:
    fig = plot_correlation_matrix(returns_dict)
    plt.show()

## 6. Executive Dashboard

4-panel unified visual summary.

In [ ]:
if returns_dict:
    fig = plot_executive_dashboard(returns_dict, summary_df)
    plt.show()